# Airline Satisfaction — robust Kaggle starter
Arun Gharami · Playground Series S6E10 · Revision 2

**Before Run All:** join the competition, attach its dataset using Add Input, and restart the session after importing this notebook. A CPU session is sufficient. No Internet, API key, or paid service is required for the fallback model.

This revision improves input discovery, missing-value handling, probability checks, and runtime. It has been tested on synthetic fixtures; the linked Kaggle error log and real competition data were not accessible during revision. No leaderboard score is claimed.

Outputs: `submission.csv`, `validation_results.csv`, and `run_metadata.json`. Save Version → Save & Run All, then submit the completed version's CSV.


In [ ]:
from pathlib import Path
import json, platform, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.metrics import roc_auc_score, log_loss

SEED = 42
VALID_SIZE = 0.20
ITERATIONS = 500
# Optional explicit folder; normally automatic discovery is sufficient.
DATA_DIR_OVERRIDE = None
THREADS = 2
# Set this only if the automatic string-label mapping raises an error.
POSITIVE_LABEL = None
OUTPUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT.mkdir(exist_ok=True)
print('Python', platform.python_version(), 'pandas', pd.__version__, 'sklearn', sklearn.__version__)


## 1. Load the attached competition files

In [ ]:
COMPETITION = 'playground-series-s6e10'
TARGET, ID = 'satisfaction', 'id'
required = ['train.csv', 'test.csv', 'sample_submission.csv']
roots = [Path('/kaggle/input'), Path('data')]
candidates = set()
for root in roots:
    if root.exists():
        for f in root.rglob('sample_submission.csv'):
            if all((f.parent / name).is_file() for name in required):
                candidates.add(f.parent)
candidates = sorted(candidates, key=str)
if DATA_DIR_OVERRIDE is not None:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)
else:
    preferred = [p for p in candidates if COMPETITION in str(p)]
    choices = preferred if preferred else candidates
    # Use column headers to disambiguate renamed competition attachments.
    choices = [p for p in choices
               if pd.read_csv(p / 'sample_submission.csv', nrows=0).columns.tolist() == [ID, TARGET]
               and TARGET in pd.read_csv(p / 'train.csv', nrows=0).columns]
    if len(choices) != 1:
        available = '\n'.join(str(p) for p in candidates) or '(none)'
        raise FileNotFoundError(
            'Competition files could not be identified uniquely. In the Kaggle editor, '
            'use Add Input / Add Data to attach Playground Series S6E10, then restart and Run All. '
            'For multiple matching folders, set DATA_DIR_OVERRIDE in the first cell. '
            'Discovered folders:\n' + available)
    DATA_DIR = choices[0]
if not all((DATA_DIR / f).is_file() for f in required):
    raise FileNotFoundError(f'{DATA_DIR} must contain {required}')
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert len(train) > 10 and len(test) > 0, 'Empty or too-small input.'
assert TARGET in train and ID in train and ID in test
assert sample.columns.tolist() == [ID, TARGET], 'Inspect the competition submission schema.'
assert len(test) == len(sample), 'Submission/test row count mismatch.'
assert train[ID].is_unique and test[ID].is_unique and sample[ID].is_unique, 'Duplicate IDs.'
assert not train[ID].isna().any() and not test[ID].isna().any() and not sample[ID].isna().any()
assert set(test[ID]) == set(sample[ID]), 'Submission/test IDs differ.'
assert not train[TARGET].isna().any(), 'Missing training labels.'
print('Data directory:', DATA_DIR)
print('Shapes:', train.shape, test.shape, sample.shape)
display(train.head())
display(train[TARGET].value_counts(dropna=False))


## 2. Check labels, features, and missing values
The positive class must represent **satisfied**. ID is excluded from model features.

In [ ]:
raw_y = train[TARGET]
classes = list(pd.unique(raw_y))
assert len(classes) == 2, f'Expected binary target; found {classes}'
if (pd.api.types.is_numeric_dtype(raw_y) or pd.api.types.is_bool_dtype(raw_y)) and set(classes) == {0, 1}:
    y = raw_y.astype(int)
    label_mapping = {'0': 0, '1': 1}
else:
    normalized = raw_y.astype(str).str.strip().str.lower()
    known = {'satisfied': 1, 'neutral or dissatisfied': 0, 'dissatisfied': 0,
             '0': 0, '1': 1}
    if POSITIVE_LABEL is not None:
        assert POSITIVE_LABEL in classes
        y = (raw_y == POSITIVE_LABEL).astype(int)
    elif set(normalized.unique()).issubset(known) and len(set(normalized.map(known))) == 2:
        y = normalized.map(known).astype(int)
    else:
        raise ValueError(f'Unknown labels {classes}. Set POSITIVE_LABEL to the label meaning satisfied.')
    label_mapping = {str(c): int(y.loc[raw_y == c].iloc[0]) for c in classes}
print('VERIFY positive class mapping:', label_mapping)
features = [c for c in train.columns if c not in [TARGET, ID]]
assert set(features) == set(test.columns) - {ID}, 'Train/test feature mismatch: inspect input.'
X = train[features].copy()
X_test = test[features].copy()
cat_cols = X.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()
num_cols = [c for c in features if c not in cat_cols]
for frame in [X, X_test]:
    for c in cat_cols:
        frame[c] = frame[c].fillna('__MISSING__').astype(str)
    frame[num_cols] = frame[num_cols].replace([np.inf, -np.inf], np.nan).astype('float32')
display(X.isna().mean().sort_values(ascending=False).head(15).rename('missing_fraction'))
print('Categorical:', cat_cols)
print('Numeric:', num_cols)
print('Duplicate feature rows:', X.duplicated().sum())
assert y.value_counts().min() >= 5, 'Too few rows in one class for stratified validation.'
assert features, 'No predictors found.'
# Record and discard completely empty numeric columns; they contain no signal.
empty_numeric = [c for c in num_cols if X[c].isna().all()]
if empty_numeric:
    print('Dropping all-missing training features:', empty_numeric)
    X = X.drop(columns=empty_numeric)
    X_test = X_test.drop(columns=empty_numeric)
    num_cols = [c for c in num_cols if c not in empty_numeric]
    features = X.columns.tolist()
assert features, 'No usable predictors found.'
train_idx, valid_idx = train_test_split(np.arange(len(X)), test_size=VALID_SIZE,
                                       stratify=y, random_state=SEED)
# This is an IID starting split. Repeated passengers/groups or temporal structure
# require a group/time split after you inspect the actual data.


## 3. Fast linear baseline
SGD logistic classification provides a CPU fallback without CatBoost. Imputation, scaling, and encoding are learned only from fitting rows. Sparse one-hot encoding avoids a dense category matrix.

In [ ]:
transformers = []
if num_cols:
    transformers.append(('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('category', OneHotEncoder(handle_unknown='ignore', dtype=np.float32), cat_cols))
preprocess = ColumnTransformer(transformers, sparse_threshold=1.0)
baseline = Pipeline([('preprocess', preprocess),
                     ('model', SGDClassifier(loss='log_loss', max_iter=100, tol=1e-3,
                                             alpha=0.0001, random_state=SEED))])
started = time.perf_counter()
baseline.fit(X.iloc[train_idx], y.iloc[train_idx])
positive_index = int(np.flatnonzero(baseline.classes_ == 1)[0])
base_valid = baseline.predict_proba(X.iloc[valid_idx])[:, positive_index]
results = [{'model': 'sgd_logistic', 'validation_auc': roc_auc_score(y.iloc[valid_idx], base_valid),
            'validation_logloss': log_loss(y.iloc[valid_idx], base_valid),
            'seconds': time.perf_counter() - started}]
display(pd.DataFrame(results))


## 4. CatBoost challenger
Uses CPU when installed. If absent, the linear baseline still produces a submission without Internet. Numeric medians come only from fitting rows. Validation chooses the model and tree count; this is a development estimate rather than an independent final test.

In [ ]:
try:
    import catboost
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None
    print('CatBoost unavailable: using the fast linear baseline for this first submission.')

boost_valid = None
best_iterations = ITERATIONS
if CatBoostClassifier is not None:
    medians = X.iloc[train_idx][num_cols].median().fillna(0)
    def fill_numeric(frame, medians):
        frame = frame.copy()
        frame[num_cols] = frame[num_cols].fillna(medians)
        return frame
    boost = CatBoostClassifier(iterations=ITERATIONS, depth=6, learning_rate=0.05,
                              loss_function='Logloss', eval_metric='AUC', random_seed=SEED,
                              thread_count=THREADS, allow_writing_files=False, verbose=False)
    started = time.perf_counter()
    boost.fit(fill_numeric(X.iloc[train_idx], medians), y.iloc[train_idx],
              cat_features=cat_cols,
              eval_set=(fill_numeric(X.iloc[valid_idx], medians), y.iloc[valid_idx]),
              early_stopping_rounds=60, verbose=100)
    boost_valid = boost.predict_proba(fill_numeric(X.iloc[valid_idx], medians))[:, 1]
    best_iterations = max(1, boost.get_best_iteration() + 1)
    results.append({'model': 'catboost', 'validation_auc': roc_auc_score(y.iloc[valid_idx], boost_valid),
                    'validation_logloss': log_loss(y.iloc[valid_idx], boost_valid),
                    'seconds': time.perf_counter() - started})
    pd.Series(boost.feature_importances_, index=features).sort_values(ascending=False).to_csv(OUTPUT / 'feature_importance.csv')
scores = pd.DataFrame(results).sort_values('validation_auc', ascending=False)
scores.to_csv(OUTPUT / 'validation_results.csv', index=False)
display(scores)
winner = scores.iloc[0]['model']
print('Selected model:', winner)


## 5. Refit on all training data and create submission
The sample submission determines the exact ID order. Only probabilities in [0, 1] are written.

In [ ]:
if winner == 'catboost':
    full_medians = X[num_cols].median().fillna(0)
    final_model = CatBoostClassifier(iterations=best_iterations, depth=6, learning_rate=0.05,
                                     loss_function='Logloss', random_seed=SEED,
                                     thread_count=THREADS, allow_writing_files=False, verbose=False)
    final_model.fit(fill_numeric(X, full_medians), y, cat_features=cat_cols)
    probabilities = final_model.predict_proba(fill_numeric(X_test, full_medians))[:, 1]
else:
    baseline.fit(X, y)
    probabilities = baseline.predict_proba(X_test)[:, int(np.flatnonzero(baseline.classes_ == 1)[0])]
assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
prediction_by_id = pd.Series(probabilities, index=test[ID])
submission = sample.copy()
submission[TARGET] = submission[ID].map(prediction_by_id)
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID].equals(sample[ID]) and not submission[TARGET].isna().any()
submission.to_csv(OUTPUT / 'submission.csv', index=False)
metadata = {'competition': COMPETITION, 'seed': SEED, 'validation_fraction': VALID_SIZE,
            'label_mapping': label_mapping, 'selected_model': winner,
            'catboost_iterations': best_iterations if winner == 'catboost' else None,
            'train_rows': len(train), 'test_rows': len(test), 'features': features, 'dropped_empty_numeric': empty_numeric,
            'python': platform.python_version(), 'pandas': pd.__version__,
            'sklearn': sklearn.__version__,
            'catboost': catboost.__version__ if CatBoostClassifier is not None else None,
            'results': scores.to_dict(orient='records')}
(OUTPUT / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
display(submission.head())
print('Ready:', OUTPUT / 'submission.csv')


## Submit and improve deliberately
Save Version → Save & Run All. When that version finishes, select its `submission.csv` using the competition submission flow. Inspect the reported score and record it alongside your local validation AUC.

Next experiments: three-fold stratified cross-validation; inspect group/duplicate leakage; compare CatBoost depth 4/6/8 under identical folds; engineer rating aggregates only after inspecting the actual columns; keep a fixed experiment log. These are future experiments, not results already achieved.

Keep competition data out of public Git repositories. Publish code, methodology, and permitted results according to the Rules. Consult the live Timeline for the final deadline.
